In [1]:
import os
import pickle
import random
import pandas as pd
from collections import defaultdict

from implicit.als import AlternatingLeastSquares
from gensim.models import Word2Vec

# Add src path if needed
import sys
sys.path.append(os.path.abspath('../src'))

# Custom imports
from cf_model import CFRecommender
from content_model import ContentRecommender
from embedding_model import EmbeddingRecommender
from hybrid_model import HybridRecommender
from reranker import Reranker
from evlauation import precision_at_k, recall_at_k, ndcg_at_k

# ------------------- Load Data -------------------
print("📁 Loading processed data...")
df_playlists = pd.read_pickle('../data/processed_playlists.pkl')
df_tracks = pd.read_pickle('../data/processed_tracks.pkl')

# ------------------- Load Models -------------------
print("📦 Loading models...")
# Load CF model
cf_model = AlternatingLeastSquares()
cf_model.load('../models/cf_model.model.npz')

cf = CFRecommender()
cf.model = cf_model
cf.prepare_matrix(df_playlists)

# Load Content model
with open('../models/content_model.pkl', 'rb') as f:
    content_data = pickle.load(f)
content = ContentRecommender()
content.tfidf_matrix = content_data['tfidf_matrix']
content.track_idx_map = {k.strip().lower(): v for k, v in content_data['track_index'].items()}

# Load Embedding model
embedding_model = Word2Vec.load('../models/embedding_model.model')
embedding = EmbeddingRecommender()
embedding.model = embedding_model

# Load Hybrid config
with open('../models/hybrid_config.pkl', 'rb') as f:
    hybrid_config = pickle.load(f)
weights = hybrid_config.get('weights', (0.4, 0.3, 0.3))

# Initialize Hybrid recommender
hybrid = HybridRecommender(cf_model=cf, content_model=content, embedding_model=embedding, weights=weights)

# Reranker
reranker = Reranker(df_tracks)

# ------------------- Evaluation -------------------
def normalize_uri(uri):
    return uri.strip().lower()

results = []
sampled_pids = df_playlists['playlist_id'].sample(10, random_state=42).tolist()

for pid in sampled_pids:
    playlist_tracks = df_playlists[df_playlists['playlist_id'] == pid]['track_uri'].tolist()
    playlist_tracks = [normalize_uri(uri) for uri in playlist_tracks]

    # Filter valid tracks for content model
    valid_tracks = [uri for uri in playlist_tracks if uri in content.track_idx_map]

    if len(valid_tracks) < 3:
        print(f"⚠️ Playlist {pid} skipped — only {len(valid_tracks)} valid tracks.")
        continue

    try:
        # Hybrid recommendation
        recommendations_with_scores = hybrid.recommend_tracks(pid, valid_tracks, top_n=20)
        if not recommendations_with_scores:
            print(f"⚠️ No recommendations returned for playlist {pid}.")
            continue

        recommended_uris = [track for track, _ in recommendations_with_scores]
        original_scores = [score for _, score in recommendations_with_scores]

        # Rerank
        reranked_uris = reranker.rerank(recommended_uris, original_scores)

        results.append({
            'pid': pid,
            'original_tracks': playlist_tracks,
            'recommended_uris': recommended_uris,
            'reranked_uris': reranked_uris
        })

    except Exception as e:
        print(f"❌ Error processing playlist {pid}: {e}")
        continue

# ------------------- Metrics -------------------
print("\n📈 Calculating metrics...")
total_p, total_r, total_ndcg = 0, 0, 0
k = 10

for res in results:
    gt = set(res['original_tracks'])
    recs = res['reranked_uris'][:k]

    total_p += precision_at_k(recs, gt, k)
    total_r += recall_at_k(recs, gt, k)
    total_ndcg += ndcg_at_k(recs, gt, k)

n = len(results)
if n == 0:
    print("⚠️ No valid evaluation results.")
else:
    print(f"\n✅ Evaluated {n} playlists")
    print(f"🎯 Precision@{k}: {total_p/n:.4f}")
    print(f"🎯 Recall@{k}:    {total_r/n:.4f}")
    print(f"🎯 NDCG@{k}:      {total_ndcg/n:.4f}")


📁 Loading processed data...
📦 Loading models...
🔄 Preparing user-item matrix...


/opt/anaconda3/lib/python3.12/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


❌ Error processing playlist 134068: 'NoneType' object is not subscriptable
❌ Error processing playlist 132455: 'NoneType' object is not subscriptable
❌ Error processing playlist 101834: 'NoneType' object is not subscriptable
❌ Error processing playlist 17466: 'NoneType' object is not subscriptable
❌ Error processing playlist 174524: 'NoneType' object is not subscriptable
❌ Error processing playlist 140460: 'NoneType' object is not subscriptable
❌ Error processing playlist 14212: 'NoneType' object is not subscriptable
❌ Error processing playlist 148254: 'NoneType' object is not subscriptable
❌ Error processing playlist 104742: 'NoneType' object is not subscriptable
❌ Error processing playlist 134322: 'NoneType' object is not subscriptable

📈 Calculating metrics...
⚠️ No valid evaluation results.


In [3]:
# 📦 Imports
import os
import sys
import pickle
import random
import pandas as pd
from collections import defaultdict

sys.path.append(os.path.abspath("../src"))

from hybrid_model import HybridRecommender
from reranker import Reranker  # assuming reranker.py exists in src

from embedding_model import EmbeddingRecommender

# --- Utility function to normalize URIs ---
def normalize_uris(uri_list):
    return [uri.lower().strip() for uri in uri_list]

# --- Evaluation metrics with normalization ---
def precision_at_k(recommended, ground_truth, k=10):
    recommended = normalize_uris(recommended[:k])
    ground_truth = normalize_uris(ground_truth)
    relevant = set(recommended) & set(ground_truth)
    return len(relevant) / k

def recall_at_k(recommended, ground_truth, k=10):
    recommended = normalize_uris(recommended[:k])
    ground_truth = normalize_uris(ground_truth)
    if not ground_truth:
        return 0.0
    relevant = set(recommended) & set(ground_truth)
    return len(relevant) / len(set(ground_truth))

def intra_list_diversity(recommended_uris, artist_map):
    # Normalize recommended URIs first
    recommended_uris = normalize_uris(recommended_uris)
    artists = [artist_map.get(uri, '') for uri in recommended_uris]
    unique_artists = len(set(artists))
    return unique_artists / len(recommended_uris) if recommended_uris else 0

# --- Load processed data ---
playlists_path = '../data/processed_playlists.pkl'
tracks_path = '../data/processed_tracks.pkl'

df_playlists = pd.read_pickle(playlists_path)
df_tracks = pd.read_pickle(tracks_path)

print("✅ Data loaded!")

# --- Build playlist_id -> track_uri mapping ---
playlist_dict = {
    pid: normalize_uris(tracks)  # normalize playlist tracks here
    for pid, tracks in zip(df_playlists['playlist_id'], df_playlists['track_uri'])
}

# --- Load trained models ---
with open('../data/models/cf_model.pkl', 'rb') as f:
    cf_model = pickle.load(f)

with open('../data/models/content_model.pkl', 'rb') as f:
    content_model = pickle.load(f)

with open('../data/models/embedding_model.pkl', 'rb') as f:
    embedding_model = pickle.load(f)

embedding_rec = EmbeddingRecommender()
embedding_rec.model = embedding_model

hybrid = HybridRecommender(cf_model=cf_model,
                           content_model=content_model,
                           embedding_model=embedding_rec)

# --- Prepare artist map and popularity for reranker ---
popularity_series = df_tracks['track_uri'].value_counts()
artist_map = pd.Series(df_tracks['artist_name'].values,
                       index=df_tracks['track_uri'].str.lower().str.strip()).to_dict()  # normalize keys here too

reranker = Reranker(track_popularity=popularity_series,
                    popularity_weight=0.7,
                    diversity_weight=0.3)
reranker.set_artist_map(artist_map)

# --- Sample playlists for evaluation ---
sample_pids = random.sample(list(playlist_dict.keys()), 10)

results = []

for pid in sample_pids:
    original_tracks = playlist_dict[pid]  # already normalized

    # Get hybrid recommendations (list of (uri, score))
    recommendations_with_scores = hybrid.recommend_tracks(pid, original_tracks, top_n=10)

    # Normalize recommended URIs
    recommended_uris = normalize_uris([uri for uri, score in recommendations_with_scores])

    # Rerank the recommendations
    reranked_uris = reranker.rerank(recommended_uris, [score for _, score in recommendations_with_scores])

    results.append({
        'pid': pid,
        'original_tracks': original_tracks,
        'recommended_uris': recommended_uris,
        'reranked_uris': reranked_uris
    })

# --- Compute evaluation metrics ---
precision_scores = []
recall_scores = []
diversity_scores = []

for r in results:
    pid = r['pid']
    gt = r['original_tracks']
    recs = r['reranked_uris']

    precision_scores.append(precision_at_k(recs, gt, k=10))
    recall_scores.append(recall_at_k(recs, gt, k=10))
    diversity_scores.append(intra_list_diversity(recs, artist_map))

print("\n📈 Evaluation Metrics (averaged over 10 playlists):")
print(f"🔹 Precision@10: {sum(precision_scores) / len(precision_scores):.4f}")
print(f"🔹 Recall@10: {sum(recall_scores) / len(recall_scores):.4f}")
print(f"🔹 Intra-list Diversity@10: {sum(diversity_scores) / len(diversity_scores):.4f}")

# --- Optional: Display example playlist & recommendations ---
def safe_display_tracks(uri_list, df_tracks):
    if isinstance(uri_list, str):
        uri_list = [uri_list]
    # Normalize uris for lookup
    normalized_uris = normalize_uris(uri_list)
    df_display = df_tracks[df_tracks['track_uri'].str.lower().str.strip().isin(normalized_uris)][['track_name', 'artist_name']]
    display(df_display.drop_duplicates())

example_result = results[0]

print(f"\n🎧 Playlist ID: {example_result['pid']}")

print("\n📀 Existing Tracks:")
safe_display_tracks(example_result['original_tracks'], df_tracks)

print("\n💡 Hybrid Recommendations (Before Reranking):")
safe_display_tracks(example_result['recommended_uris'], df_tracks)

print("\n🎯 Final Reranked Recommendations:")
safe_display_tracks(example_result['reranked_uris'], df_tracks)


✅ Data loaded!
⚠️ No valid tracks found in playlist for content-based recommendation.
⚠️ No valid tracks found in playlist for content-based recommendation.
⚠️ No valid tracks found in playlist for content-based recommendation.
⚠️ No valid tracks found in playlist for content-based recommendation.
⚠️ No valid tracks found in playlist for content-based recommendation.
⚠️ No valid tracks found in playlist for content-based recommendation.
⚠️ No valid tracks found in playlist for content-based recommendation.
⚠️ No valid tracks found in playlist for content-based recommendation.
⚠️ No valid tracks found in playlist for content-based recommendation.
⚠️ No valid tracks found in playlist for content-based recommendation.

📈 Evaluation Metrics (averaged over 10 playlists):
🔹 Precision@10: 0.0000
🔹 Recall@10: 0.0000
🔹 Intra-list Diversity@10: 1.0000

🎧 Playlist ID: 173979

📀 Existing Tracks:


,track_name,artist_name



💡 Hybrid Recommendations (Before Reranking):


,track_name,artist_name
0,Lose Control (feat. Ciara & Fat Man Scoop),Missy Elliott
2186,House Party,Sam Hunt



🎯 Final Reranked Recommendations:


,track_name,artist_name
0,Lose Control (feat. Ciara & Fat Man Scoop),Missy Elliott
2186,House Party,Sam Hunt
